# 02 · Data quality — Kennzahlen statt Plots

**Report-Unterkapitel:** Data Understanding → *Data quality*
**Datenquelle:** `data/raw/` (unveränderte Lieferung)

**Entscheidung: Für dieses Unterkapitel wird bewusst keine Abbildung gebaut.**
Geprüft wurden die üblichen Kandidaten — jeder scheitert an dem, was real in
den Daten steht:

| Kandidat | Befund in den echten Daten | Konsequenz |
|---|---|---|
| Missing-Value-Matrix (Haushalt × Zeit) | Ziel-Missingness konzentriert sich auf **5 von 156** Haushalten (Top 5 = 100 % der 4.293 Lücken) | Matrix zeigt 3 volle Zeilen + Rauschen — ein Satz trägt das besser |
| Vollständigkeit über Zeit | identisch mit dem Coverage-Panel aus Notebook 01 | Querverweis statt Doppelplot |
| Regelmäßigkeit der Zeitstempel | alle 88.791 Timestamps exakt `23:59:59+00:00`, 0 Duplikate | nichts zu zeigen |
| Wetter-Vollständigkeit | Stundenraster aller 8 Stationen lückenlos, NaN < 1,1 % | nichts zu zeigen |
| Ausreißer-/Verteilungsplots | gehört inhaltlich zur EDA (Notebook 03) | dort behandelt |
| Kanal-Verfügbarkeit | je Haushalt quasi binär (liefert / liefert nicht) | kompakte Tabelle reicht |

Dieses Notebook berechnet stattdessen die Data-Quality-Kennzahlen reproduzierbar
aus den Rohdaten (*compute, don't transcribe*) und exportiert sie als
Markdown-Tabellen nach `docs/data_quality_summary.md`.

In [1]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Repo-Wurzel relativ zum Notebook (notebooks/data_understanding/ -> Root)
ROOT = Path("..") / ".."
RAW = ROOT / "data" / "raw"
IMAGES = ROOT / "images" / "data_understanding"
IMAGES.mkdir(parents=True, exist_ok=True)

# Farb- und Textrollen (validierte Referenzpalette, Light Mode)
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, BASE, SURFACE = "#e1e0d9", "#c3c2b7", "#ffffff"

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "Arial", "DejaVu Sans"],
    "text.color": INK, "axes.labelcolor": INK2, "axes.edgecolor": BASE,
    "xtick.color": BASE, "ytick.color": BASE,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-",
    "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.8,
    "axes.labelsize": 10.5, "xtick.labelsize": 9.5, "ytick.labelsize": 9.5,
    "legend.frameon": False, "legend.fontsize": 9.5,
    "figure.dpi": 110,
})


def export(fig, name):
    """Jede Abbildung als Vektor-PDF (Report) und PNG (Vorschau) ablegen."""
    fig.savefig(IMAGES / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(IMAGES / f"{name}.png", bbox_inches="tight", dpi=200)
    print(f"exportiert: images/data_understanding/{name}.(pdf|png)")

In [2]:
# Rohdaten laden: 156 Haushalts-CSVs (Semikolon-getrennt), IDs als String
files = sorted((RAW / "smart_meter_daily").glob("*.csv"))
sm = pd.concat(
    [pd.read_csv(f, sep=";", dtype={"Household_ID": str}) for f in files],
    ignore_index=True,
)
sm["date"] = (
    pd.to_datetime(sm["Timestamp"], utc=True).dt.normalize().dt.tz_localize(None)
)
TGT = "kWh_received_Total"
print(f"{len(files)} Dateien | {len(sm):,} Zeilen | {sm['Household_ID'].nunique()} Haushalte")
print(f"Zeitraum: {sm['date'].min().date()} bis {sm['date'].max().date()}")

156 Dateien | 88,791 Zeilen | 156 Haushalte
Zeitraum: 2018-11-02 bis 2024-03-20


In [3]:
# 1) Strukturelle Regelmäßigkeit
times = pd.to_datetime(sm["Timestamp"], utc=True).dt.strftime("%H:%M:%S")
chan = [c for c in sm.columns if c.startswith(("kWh", "kvarh"))]
structural = {
    "Zeitstempel exakt 23:59:59 UTC": f"{(times == '23:59:59').mean():.1%}",
    "Duplikate (Household_ID, date)": int(sm.duplicated(["Household_ID", "date"]).sum()),
    "negative Messwerte (alle 10 Kanäle)": int((sm[chan] < 0).sum().sum()),
    "leere Household_ID / Timestamp": int(sm[["Household_ID", "Timestamp"]].isna().sum().sum()),
}
for k, v in structural.items():
    print(f"{k}: {v}")

Zeitstempel exakt 23:59:59 UTC: 100.0%
Duplikate (Household_ID, date): 0
negative Messwerte (alle 10 Kanäle): 0
leere Household_ID / Timestamp: 0


In [4]:
# 2) Vollständigkeit der Zielvariable — und ihre Konzentration
n_missing = sm[TGT].isna().sum()
print(f"Zielwert fehlt in {n_missing:,} von {len(sm):,} gelieferten Zeilen "
      f"({n_missing / len(sm):.2%})")

conc = (
    sm[sm[TGT].isna()].groupby("Household_ID").size()
    .sort_values(ascending=False).rename("fehlende Zielwerte").to_frame()
)
conc["Anteil an allen Lücken"] = (conc["fehlende Zielwerte"] / n_missing).map("{:.1%}".format)
n_rows_hh = sm.groupby("Household_ID").size()
conc["Anteil an eigener Historie"] = (
    conc["fehlende Zielwerte"] / n_rows_hh[conc.index]
).map("{:.1%}".format)
print(conc.to_string())
print(f"\nHaushalte mit mindestens einer Ziellücke: {len(conc)} von 156")

Zielwert fehlt in 4,293 von 88,791 gelieferten Zeilen (4.83%)
              fehlende Zielwerte Anteil an allen Lücken Anteil an eigener Historie
Household_ID                                                                      
996610                      1823                  42.5%                     100.0%
816910                       957                  22.3%                      92.6%
747511                       770                  17.9%                     100.0%
768498                       734                  17.1%                     100.0%
610891                         9                   0.2%                       5.8%

Haushalte mit mindestens einer Ziellücke: 5 von 156


In [5]:
# 3) Kalenderlücken je Haushalt
g = sm.groupby("Household_ID")["date"]
per_hh = pd.DataFrame({"first": g.min(), "last": g.max(), "n_rows": g.size()})
per_hh["span"] = (per_hh["last"] - per_hh["first"]).dt.days + 1
per_hh["gap_days"] = per_hh["span"] - per_hh["n_rows"]
gaps = per_hh[per_hh.gap_days > 0].sort_values("gap_days", ascending=False)
print(f"Haushalte mit Kalenderlücken: {len(gaps)} von 156 | "
      f"fehlende Tage gesamt: {int(gaps.gap_days.sum()):,} | Maximum: {int(gaps.gap_days.max())}")
print(gaps.head(5)[["first", "last", "n_rows", "gap_days"]].to_string())

Haushalte mit Kalenderlücken: 41 von 156 | fehlende Tage gesamt: 1,356 | Maximum: 333
                  first       last  n_rows  gap_days
Household_ID                                        
861116       2018-11-02 2024-03-20    1633       333
611629       2018-11-02 2024-03-20    1720       246
699801       2019-03-02 2024-03-20    1661       185
881223       2020-02-07 2020-11-01     133       136
712718       2018-11-02 2021-01-06     710        87


In [6]:
# 4) Kanal-Verfügbarkeit: je Haushalt quasi binär
fill = sm.groupby("Household_ID")[chan].apply(lambda d: d.notna().mean())
channels = pd.DataFrame({
    "Haushalte mit Kanal": (fill > 0).sum(),
    "davon > 95 % gefüllt": (fill > 0.95).sum(),
    "Zeilen-Füllgrad gesamt": sm[chan].notna().mean().map("{:.1%}".format),
}).rename_axis("Kanal")
print(channels.to_string())
print("\nLesart: Ob ein Kanal geliefert wird, ist eine Haushaltseigenschaft "
      "(Zählerkonfiguration), keine zufällige Messlücke.")

                                    Haushalte mit Kanal  davon > 95 % gefüllt Zeilen-Füllgrad gesamt
Kanal                                                                                               
kWh_received_Total                                  153                   151                  95.2%
kWh_received_HeatPump                                10                     9                   7.5%
kWh_received_Other                                    7                     6                   4.9%
kWh_returned_Total                                   37                     9                  16.0%
kvarh_received_capacitive_Total                     101                    91                  56.1%
kvarh_received_capacitive_HeatPump                    5                     3                   2.6%
kvarh_received_capacitive_Other                       4                     3                   1.9%
kvarh_received_inductive_Total                       97                    66              

In [7]:
# 5) Wetterdaten: Raster, NaN-Raten, strukturell fehlende Kanäle
rows = []
for f in sorted((RAW / "weather_data_hourly").glob("*.csv")):
    w = pd.read_csv(f, sep=";")
    w["Timestamp"] = pd.to_datetime(w["Timestamp"], utc=True)
    full = pd.date_range(w["Timestamp"].min(), w["Timestamp"].max(), freq="h")
    vals = w.drop(columns=["Weather_ID", "Timestamp"])
    empty_cols = [c for c in vals.columns if vals[c].isna().all()]
    partial = vals.drop(columns=empty_cols).isna().mean()
    rows.append({
        "Station": f.stem, "Zeilen": len(w),
        "Raster lückenlos": len(full) == len(w),
        "fehlende Kanäle": len(empty_cols),
        "max. NaN-Rate übrige Kanäle": f"{partial.max():.2%}",
    })
weather_q = pd.DataFrame(rows).set_index("Station")
print(weather_q.to_string())
print("\nStrukturell ohne Sunshine- und Pressure-Kanäle: ceOxS, HbsbG, sV3mR "
      "(Sensorik fehlt, keine Messlücke).")

         Zeilen  Raster lückenlos  fehlende Kanäle max. NaN-Rate übrige Kanäle
Station                                                                       
8jB       45264              True                0                       0.35%
ceOxS     45264              True                4                       2.57%
HbsbG     45264              True                4                       1.87%
Hg        45264              True                0                       0.11%
MqO       45264              True                1                       0.11%
sV3mR     45264              True                4                       0.75%
wDD       45264              True                0                       0.11%
z6I       45264              True                0                       0.37%

Strukturell ohne Sunshine- und Pressure-Kanäle: ceOxS, HbsbG, sV3mR (Sensorik fehlt, keine Messlücke).


In [8]:
# 6) UTC-Tagesgrenze: Was kostet das UTC-Fenster gegenüber Europe/Berlin?
# D-01 legt die Tagesgrenze auf die UTC-Mitternacht. Messbar ist der Effekt
# nur auf der Wetterseite (Stundendaten vorhanden); die Last ist ausschließlich
# als Tageswert geliefert und lässt sich nicht umaggregieren.
wh = pd.concat(
    [pd.read_csv(f, sep=";", usecols=["Weather_ID", "Timestamp", "Temperature_avg_hourly"])
     for f in sorted((RAW / "weather_data_hourly").glob("*.csv"))],
    ignore_index=True,
)
wh["Timestamp"] = pd.to_datetime(wh["Timestamp"], utc=True)
berlin = wh["Timestamp"].dt.tz_convert("Europe/Berlin")

# Lokale Uhrzeit des Zählerstempels 23:59:59Z: 00:59:59 (CET) bzw. 01:59:59 (CEST)
off_h = (berlin.dt.tz_localize(None) - wh["Timestamp"].dt.tz_localize(None)).dt.total_seconds() / 3600
dst_share = float((off_h == 2).mean())

wh["day_utc"] = wh["Timestamp"].dt.tz_localize(None).dt.normalize()
wh["day_berlin"] = berlin.dt.tz_localize(None).dt.normalize()
t_utc = wh.groupby(["Weather_ID", "day_utc"])["Temperature_avg_hourly"].mean().rename("t_utc")
t_ber = wh.groupby(["Weather_ID", "day_berlin"])["Temperature_avg_hourly"].mean().rename("t_berlin")
# unvollständige Randtage des Berlin-Fensters je Station abschneiden
t_ber = t_ber.groupby(level=0).apply(lambda s: s.droplevel(0).iloc[1:-1])
t_utc.index.names = t_ber.index.names = ["Weather_ID", "day"]
both = pd.concat([t_utc, t_ber], axis=1, join="inner").dropna()

diff = both["t_utc"] - both["t_berlin"]
hdd_u, hdd_b = (15 - both["t_utc"]).clip(lower=0), (15 - both["t_berlin"]).clip(lower=0)
hdd_diff = hdd_u - hdd_b
utc_boundary = {
    "Stations-Tage im Vergleich": f"{len(both):,}",
    "Zählerstempel 23:59:59 UTC in Lokalzeit": f"00:59:59 (CET, {1 - dst_share:.1%}) / 01:59:59 (CEST, {dst_share:.1%})",
    "Tagesmittel-Temperatur: r (UTC- vs. Berlin-Fenster)": f"{both['t_utc'].corr(both['t_berlin']):.4f}",
    "Tagesmittel-Temperatur: mittlere Differenz": f"{diff.mean():+.3f} K",
    "Tagesmittel-Temperatur: mittlere absolute Differenz": f"{diff.abs().mean():.3f} K",
    "Tagesmittel-Temperatur: P95 / Maximum der absoluten Differenz": f"{diff.abs().quantile(0.95):.2f} K / {diff.abs().max():.2f} K",
    "hdd_15: mittlere absolute Differenz": f"{hdd_diff.abs().mean():.3f} K",
    "hdd_15: Anteil Tage mit absoluter Differenz > 0,5 K": f"{(hdd_diff.abs() > 0.5).mean():.2%}",
}
for k, v in utc_boundary.items():
    print(f"{k}: {v}")
print("\nLesart: Die Fensterwahl ändert die Tagesaggregate praktisch nicht — "
      "weder Niveau (mittlere Differenz ≈ 0) noch Struktur (r ≈ 1). Für die "
      "Last ist der Effekt nicht messbar: sie liegt nur als Tageswert vor (D-01).")

Stations-Tage im Vergleich: 15,070
Zählerstempel 23:59:59 UTC in Lokalzeit: 00:59:59 (CET, 43.2%) / 01:59:59 (CEST, 56.8%)
Tagesmittel-Temperatur: r (UTC- vs. Berlin-Fenster): 0.9996
Tagesmittel-Temperatur: mittlere Differenz: +0.000 K
Tagesmittel-Temperatur: mittlere absolute Differenz: 0.145 K
Tagesmittel-Temperatur: P95 / Maximum der absoluten Differenz: 0.41 K / 2.20 K
hdd_15: mittlere absolute Differenz: 0.094 K
hdd_15: Anteil Tage mit absoluter Differenz > 0,5 K: 1.25%

Lesart: Die Fensterwahl ändert die Tagesaggregate praktisch nicht — weder Niveau (mittlere Differenz ≈ 0) noch Struktur (r ≈ 1). Für die Last ist der Effekt nicht messbar: sie liegt nur als Tageswert vor (D-01).


In [9]:
# 7) Export der Kennzahlen als Markdown nach docs/
def md_table(df):
    df = df.reset_index()
    head = "| " + " | ".join(map(str, df.columns)) + " |"
    sep = "|" + "|".join("---" for _ in df.columns) + "|"
    body = "\n".join("| " + " | ".join(map(str, r)) + " |" for r in df.to_numpy())
    return f"{head}\n{sep}\n{body}"

doc = f"""# Data quality — berechnete Kennzahlen

_Automatisch erzeugt von `notebooks/data_understanding/02_data_quality.ipynb`
aus `data/raw/`. Zahlen werden bei jedem Lauf neu berechnet._

## Strukturelle Regelmäßigkeit

{md_table(pd.Series(structural, name="Wert").to_frame().rename_axis("Prüfung"))}

## Zielvariable `kWh_received_Total`

Fehlend in {n_missing:,} von {len(sm):,} gelieferten Zeilen
({n_missing / len(sm):.2%}) — konzentriert auf {len(conc)} von 156 Haushalten:

{md_table(conc)}

## Kalenderlücken

{len(gaps)} von 156 Haushalten haben innere Kalenderlücken
({int(gaps.gap_days.sum()):,} fehlende Tage, Maximum {int(gaps.gap_days.max())}
Tage am Stück). Sichtbar in `images/data_understanding/01_delivered_recording_coverage`.

## Kanal-Verfügbarkeit (je Haushalt quasi binär)

{md_table(channels)}

## Wetterdaten (8 Stationen, stündlich)

{md_table(weather_q)}

## UTC-Tagesgrenze (D-01)

Messung auf den Wetterstundendaten — die Last ist nur als Tageswert geliefert,
dort ist die Fensterwahl nicht nachmessbar:

{md_table(pd.Series(utc_boundary, name="Wert").to_frame().rename_axis("Kennzahl"))}
"""
out = ROOT / "docs" / "data_quality_summary.md"
out.write_text(doc, encoding="utf-8")
print(f"geschrieben: {out.resolve()}")

geschrieben: C:\Users\mamac\Documents\Uni\Lectures\Semester_02\Data_Analytics_in_Applications\Coding\TUM_SS26_v3\docs\data_quality_summary.md


## Fazit für das Report-Unterkapitel

Die Datenqualität dieses Datensatzes ist **strukturell**, nicht diffus: Lücken
sind entweder eine Haushaltseigenschaft (Kanal nicht konfiguriert, Zähler tot)
oder eine Stationseigenschaft (Sensor fehlt) — fast nie zufälliges Rauschen.
Genau deshalb trägt eine kompakte Tabelle die Information besser als jede
Missing-Value-Grafik: Es gibt keine Verteilung zu zeigen, nur wenige klar
benennbare Fälle. Einziger visueller Anker des Unterkapitels bleibt der
Querverweis auf die Coverage-Abbildung aus Notebook 01, in der Kalenderlücken
und die toten Haushalte bereits sichtbar sind.